# 실습 2 — Vector Search 2.0 상품 검색 엔진 직접 돌려보기

배포 중인 쇼핑 에이전트(`app/embedding_vector.py`)가 런타임에 수행하는 것과 **동일한 API 호출**을, `install.sh`가 만들어 둔 컬렉션 `amazon-product-768-compact`(768차원 dense 필드 2개 + ScaNN 인덱스 2개)에 대해 실행합니다.

> [!IMPORTANT]
> 이 노트북은 컬렉션·인덱스를 만들지 않습니다. 시작 전에 `part2/README.md`의 Cloud Run 배포 명령을 먼저 실행하세요 — 빌드(약 5분)가 이 실습과 병렬로 돕니다.

## 1. 클라이언트 초기화 및 컬렉션 핸들

앱과 같은 클라이언트 4종을 만듭니다 — 질의 임베딩(`genai`), 벡터·배치 검색(`DataObjectSearch`), 개별 조회(`DataObject`), 리랭킹(`Rank`).

In [ ]:
import io
import statistics
import urllib.request
from html import escape
from pathlib import Path
from time import perf_counter

import google.auth
from google import genai
from google.genai import types
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import HTML, display
from PIL import Image

_, PROJECT_ID = google.auth.default()

# ── app/common.py 와 동일한 값 ──────────────────────────────────────────
LOCATION = "asia-northeast1"
COLLECTION_ID = "amazon-product-768-compact"
COLLECTION_NAME = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
IMAGE_SERVER = "https://thumbnail.aidemo.dev"

# ── app/embedding_vector.py 와 동일한 값 ────────────────────────────────
EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 768
TEXT_FIELD = "text_embedding"
IMAGE_FIELD = "image_embedding"
RANKING_CONFIG = f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config"
TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]
IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]

embedding_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
search_client = vectorsearch.DataObjectSearchServiceClient()
data_client = vectorsearch.DataObjectServiceClient()
rank_client = discoveryengine.RankServiceClient()

print("PROJECT_ID :", PROJECT_ID)
print("COLLECTION :", COLLECTION_NAME)

## 2. 백그라운드 인덱싱 완료 확인

`install.sh`가 백그라운드로 띄운 `session2_index_builder.py`의 LRO 4개 — 컬렉션 생성 → `ImportDataObjects` → ScaNN 인덱스 2개 — 가 모두 `done`인지 확인합니다.

In [ ]:
!gcloud vector-search operations list --location=asia-northeast1

In [ ]:
# 컬렉션에 실제로 상품이 몇 건 적재되었는지 확인합니다. (벡터 없이 집계만 수행)
try:
    response = search_client.aggregate_data_objects(
        vectorsearch.AggregateDataObjectsRequest(parent=COLLECTION_NAME, aggregate="COUNT")
    )
    # aggregate_results 는 Struct 리스트로 돌아옵니다. dict 로 풀어야 읽을 수 있습니다.
    rows = [dict(row) for row in response.aggregate_results]
    print("적재된 상품 수 :", rows)
except Exception as exc:  # 임포트가 아직 진행 중이면 여기로 들어옵니다.
    print("❌ 집계 실패:", exc)
    print()
    print("확인 순서:")
    print("  1) 위 2단계의 operations 목록에서 임포트 작업이 done: true 인지")
    print("  2) 터미널에서  tail -30 ~/smx-multimodal-agent/index_builder.log")
    print("  3) 그래도 비어 있으면  bash install.sh  를 다시 실행")


## 3. 컬렉션 스키마 확인 — dense 벡터 필드 2개

`text_embedding`은 상품명+설명문을, `image_embedding`은 대표 이미지를 임베딩한 필드입니다.
Gemini Embedding 2가 둘을 같은 벡터 공간에 넣으므로, 질의 벡터 하나를 두 필드 모두에 던질 수 있습니다.

In [ ]:
try:
    service_client = vectorsearch.VectorSearchServiceClient()
    collection = service_client.get_collection(name=COLLECTION_NAME)
    print("── data_schema (검색 결과로 돌려받을 수 있는 데이터 필드) ──")
    print(collection.data_schema)
    print("── vector_schema (검색 대상 벡터 필드) ──")
    print(collection.vector_schema)
except Exception as exc:
    print("컬렉션 조회 실패:", exc)

## 4. 상품 카탈로그 프리뷰 + 공용 헬퍼 정의

앞으로 계속 쓸 헬퍼를 정의합니다. 대응하는 앱 함수는 각 docstring에 적혀 있습니다.

> 이 컬렉션에는 서버측 자동 임베딩(`vertex_embedding_config`)이 없습니다. 클라이언트가 벡터를 만들어 넣는 `VectorSearch`(bring-your-own-vector) 방식을 씁니다.

In [ ]:
def embed(text: str | None = None, image: bytes | None = None) -> list[float]:
    """app/embedding_vector.py 의 _embed_with_gemini_embedding_2() 와 동일."""
    contents = text if text is not None else types.Part.from_bytes(data=image, mime_type="image/jpeg")
    response = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
    )
    return list(response.embeddings[0].values)


def to_item(result) -> dict:
    """app/embedding_vector.py 의 _search_result_to_dict() 와 동일."""
    obj = result.data_object
    item_id = obj.data_object_id or obj.name.split("/")[-1]
    return {
        "id": item_id,
        "name": str(obj.data.get("name", "")),
        "description": str(obj.data.get("description", "")),
        "score": result.distance,
    }


def vector_search(embedding, search_field, top_k=8, metadata_filter=None) -> list[dict]:
    """app/embedding_vector.py 의 _text/_image_similarity_collection_search() 와 동일."""
    clause_kwargs = {
        "search_field": search_field,
        "vector": vectorsearch.DenseVector(values=embedding),
        "top_k": top_k,
        "output_fields": vectorsearch.OutputFields(data_fields=["name", "description"]),
    }
    if metadata_filter is not None:  # 9단계에서 사용합니다.
        clause_kwargs["filter"] = metadata_filter
    request = vectorsearch.SearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        vector_search=vectorsearch.VectorSearch(**clause_kwargs),
    )
    response = search_client.search_data_objects(request)
    return [to_item(result) for result in response.results]


def dedupe(items):
    """상품명이 같은 항목을 하나만 남깁니다.

    Amazon 카탈로그에는 색상·사이즈 변형이 서로 다른 ID로 들어 있어서, 그대로 그리면
    상위 10건 중 8건이 같은 이름으로 보입니다. 순위가 바뀌어도 화면이 안 바뀐 것처럼
    오해되므로 표시 단계에서만 걸러냅니다. (검색 자체는 원본 결과를 그대로 씁니다.)
    """
    seen, unique = set(), []
    for item in items:
        key = item["name"].strip().lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def render(items, title="", limit=8):
    """검색 결과를 썸네일 그리드로 표시합니다."""
    items = dedupe(items)
    cards = []
    for rank, item in enumerate(items[:limit], 1):
        cards.append(
            "<div style='width:148px;margin:6px;font-size:11px;text-align:center'>"
            "<img src='{}/{}.webp' style='width:140px;height:140px;object-fit:contain;"
            "background:#fff;border:1px solid #eee'>"
            "<div><b>{}.</b> {}</div><div style='color:#888'>{:.4f}</div></div>".format(
                IMAGE_SERVER, item["id"], rank, escape(item["name"])[:64], item["score"]
            )
        )
    display(HTML(
        "<b>{}</b><div style='display:flex;flex-wrap:wrap'>{}</div>".format(
            escape(title), "".join(cards))))


preview = vector_search(embed(text="summer floral dress"), TEXT_FIELD, top_k=8)
render(preview, "카탈로그 프리뷰 — 'summer floral dress'")

## 5. 텍스트 질의 ➔ `text_embedding` 필드 검색

`find_items` 툴이 받은 영어 쿼리를 처리하는 경로입니다. 두 번째 질의에는 `thermos`·`mug` 같은 단어가 없지만 보온 용기류가 상위에 옵니다.

In [ ]:
QUERIES = [
    "waterproof hiking shoes for rainy trails",
    "something that keeps my coffee hot on the desk all morning",
]

text_results = []
for query in QUERIES:
    embed_started = perf_counter()
    query_vector = embed(text=query)
    embed_ms = (perf_counter() - embed_started) * 1000

    search_started = perf_counter()
    results = vector_search(query_vector, TEXT_FIELD, top_k=8)
    search_ms = (perf_counter() - search_started) * 1000

    print("query={!r}  embed_ms={:.1f}  search_ms={:.1f}  results={}".format(
        query, embed_ms, search_ms, len(results)))
    render(results, "text_embedding ← " + query)
    if not text_results:
        text_results = results

## 6. 이미지 질의 ➔ `image_embedding` 필드 검색 (크로스모달)

같은 이미지 벡터 하나를 ① `image_embedding`(외형 유사, 질의 상품 자신이 1등인 것이 정상) ② `text_embedding`(이미지 벡터로 설명문을 찾는 크로스모달) 두 필드에 각각 던집니다. 앱의 카메라 경로가 ①입니다.

두 결과가 서로 다르다는 것이 다음 단계에서 RRF로 융합하는 이유입니다.

In [ ]:
def fetch_jpeg(url: str) -> bytes:
    """썸네일을 내려받아 JPEG 바이트로 변환합니다 (앱이 카메라에서 받는 형식과 동일)."""
    with urllib.request.urlopen(url) as response:
        raw = response.read()
    buffer = io.BytesIO()
    Image.open(io.BytesIO(raw)).convert("RGB").save(buffer, format="JPEG")
    return buffer.getvalue()


seed = text_results[0]
seed_url = "{}/{}.webp".format(IMAGE_SERVER, seed["id"])
print("질의 이미지 :", seed["name"])
display(HTML("<img src='{}' width='180' style='border:1px solid #eee'>".format(seed_url)))

image_vector = embed(image=fetch_jpeg(seed_url))
print("이미지 질의 벡터 차원 :", len(image_vector))

render(vector_search(image_vector, IMAGE_FIELD, top_k=8), "① image_embedding ← 이미지 벡터 (생김새)")
render(vector_search(image_vector, TEXT_FIELD, top_k=8), "② text_embedding ← 이미지 벡터 (크로스모달)")

## 7. [핵심] 하이브리드 검색과 RRF 가중치 실험

`batch_search_data_objects`는 검색 절 여러 개를 한 번의 왕복으로 실행하고 서버 내장 RRF로 융합합니다.

$$\text{score}(d) = \sum_{i} w_i \cdot \frac{1}{k + \text{rank}_i(d)}$$

Part 1의 `alpha`에 대응하는 것이 `weights`이며, **절 순서 = 가중치 순서**입니다.

| | 1번 절 (`text_embedding`) | 2번 절 (`image_embedding`) |
| :--- | :--- | :--- |
| `TEXT_QUERY_HYBRID_WEIGHTS` | **1.35** | 0.65 |
| `IMAGE_QUERY_HYBRID_WEIGHTS` | 0.65 | **1.35** |

같은 질의 벡터로 가중치만 뒤집어 순위 변동을 비교합니다.

In [ ]:
def hybrid_search(embedding, weights, top_k=20) -> list[dict]:
    """app/embedding_vector.py 의 _hybrid_collection_search() 와 동일한 요청."""
    request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        searches=[
            vectorsearch.Search(  # 1번 절 → weights[0]
                vector_search=vectorsearch.VectorSearch(
                    search_field=TEXT_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
            vectorsearch.Search(  # 2번 절 → weights[1]
                vector_search=vectorsearch.VectorSearch(
                    search_field=IMAGE_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=weights)
            ),
            output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
            top_k=top_k,
        ),
    )
    response = search_client.batch_search_data_objects(request)
    fused = response.results[0].results if response.results else []
    return [to_item(result) for result in fused]


def compare(left_title, left, right_title, right, limit=8):
    """두 결과 리스트를 나란히 놓고 순위 변동을 표시합니다."""
    left, right = dedupe(left), dedupe(right)
    left_rank = {item["id"]: i for i, item in enumerate(left, 1)}
    right_rank = {item["id"]: i for i, item in enumerate(right, 1)}

    def badge(item, rank, other):
        previous = other.get(item["id"])
        if previous is None:
            return "<span style='color:#c0392b'>NEW</span>"
        if previous == rank:
            return "<span style='color:#aaa'>=</span>"
        if previous > rank:
            return "<span style='color:#1e8449'>▲{}</span>".format(previous - rank)
        return "<span style='color:#2471a3'>▼{}</span>".format(rank - previous)

    def cells(items, rank, other):
        if rank > len(items):
            return "<td></td><td></td>"
        item = items[rank - 1]
        return ("<td style='padding:4px'><img src='{}/{}.webp' width='52' "
                "style='object-fit:contain;background:#fff'></td>"
                "<td style='padding:4px;font-size:11px'>{} {}</td>").format(
                    IMAGE_SERVER, item["id"], escape(item["name"])[:46], badge(item, rank, other))

    rows = []
    for rank in range(1, limit + 1):
        rows.append("<tr><td style='padding:4px;color:#888'>{}</td>{}{}</tr>".format(
            rank, cells(left, rank, right_rank), cells(right, rank, left_rank)))
    display(HTML(
        "<table style='border-collapse:collapse'>"
        "<tr><th></th><th colspan='2' style='padding:6px'>{}</th>"
        "<th colspan='2' style='padding:6px'>{}</th></tr>{}</table>".format(
            escape(left_title), escape(right_title), "".join(rows))))


text_weighted = hybrid_search(image_vector, TEXT_QUERY_HYBRID_WEIGHTS)
image_weighted = hybrid_search(image_vector, IMAGE_QUERY_HYBRID_WEIGHTS)

compare(
    "TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]", text_weighted,
    "IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]", image_weighted,
)
print("▲▼ 는 반대편 목록 대비 순위 변동, NEW 는 반대편 상위 8위 안에 없던 상품입니다.")

> 가중치 한쪽을 0으로 주면(`[2.0, 0.0]`, `[0.0, 2.0]`) 해당 절이 무력화되어 단일 필드 검색과 같아집니다. 어느 극단도 정답이 아닌 것이 하이브리드를 쓰는 이유입니다.

## 8. Ranking API 리랭킹

벡터 검색은 재현율(recall)을, Ranking API는 정밀도(precision)를 담당합니다. 앞단에서 100건을 근사로 확보하고, Ranking API가 질의와 교차 인코딩해 재정렬합니다. `find_items`가 `ranking_query`를 따로 받는 이유입니다.

In [ ]:
def rank_results(query: str, results: list[dict]) -> list[dict]:
    """app/embedding_vector.py 의 _rank_results() 와 동일 (원본은 리스트를 제자리 정렬)."""
    if not results or not query:
        return results
    records = [
        discoveryengine.RankingRecord(
            id=item["id"], title=item["name"], content=item.get("description", "")
        )
        for item in results
    ]
    response = rank_client.rank(
        request=discoveryengine.RankRequest(
            ranking_config=RANKING_CONFIG,
            query=query,
            records=records,
            top_n=len(records),
        )
    )
    scores = {record.id: record.score for record in response.records}
    ranked = [dict(item, score=scores.get(item["id"], 0.0)) for item in results]
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked


# 6단계의 질의 이미지는 QUERIES[0] 검색 결과에서 골랐습니다.
# 리랭킹 질의도 같은 의도여야 순위 변동이 의미를 갖습니다.
RANKING_QUERY = QUERIES[0]

reranked = rank_results(RANKING_QUERY, text_weighted)
compare("RRF 융합 직후", text_weighted, "Ranking API 리랭킹 후 — " + RANKING_QUERY, reranked)
print("점수 스케일도 바뀝니다: RRF 융합 점수 → Ranking API 관련도 점수(0~1).")

## 9. 메타데이터 필터 결합

VS2 필터는 MongoDB 스타일 JSON이며 검색 절마다 지정합니다 (`$eq`, `$ne`, `$lt`, `$gt`, `$in`, `$and`, `$or`).

```python
filter={"$and": [{"category": {"$eq": "Shorts"}}, {"retail_price": {"$lt": 30}}]}
```

> 이 컬렉션의 `data_schema`는 `name`, `description` 둘뿐이라 예제는 `name`으로 거릅니다. 실서비스라면 필터 대상 속성을 `data_schema`에 넣고 인덱스 생성 시 `filter_fields=[...]`로 선언해야 대규모에서도 빠릅니다.

In [ ]:
allowed_names = [item["name"] for item in preview[:3]]
print("필터로 허용할 상품 3건:")
for name in allowed_names:
    print("  -", name[:70])

filtered = vector_search(
    embed(text="summer floral dress"),
    TEXT_FIELD,
    top_k=8,
    metadata_filter={"name": {"$in": allowed_names}},
)
print("\n필터 적용 후 결과 수:", len(filtered), "(유사도와 무관하게 허용 목록 밖 상품은 제외됩니다)")
render(filtered, "text_embedding + filter={'name': {'$in': [...]}}")

## 10. ANN(ScaNN) vs kNN — 코드는 그대로, 속도만 바뀐다

| | Part 1 미디어 컬렉션 | Part 2 상품 컬렉션 |
| :--- | :--- | :--- |
| 인덱스 | 없음 | ScaNN 2개 |
| 검색 방식 | kNN 완전탐색 | ANN 근사탐색 |
| 데이터 규모 | 약 1,000건 | 수만 건 |
| 질의 코드 | `SearchDataObjectsRequest(...)` | **완전히 동일** |

인덱스는 별도의 검색 엔드포인트가 아닙니다. 검색은 항상 컬렉션을 향하고, 질의 필드에 인덱스가 있으면 서버가 사용합니다.

In [ ]:
def find_file(*candidates) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(candidates)


def show_function(path: Path, name: str) -> None:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith("def " + name + "("))
    end = start + 1
    while end < len(lines):
        line = lines[end]
        if line.strip() and not line[:1].isspace():   # 들여쓰기가 끝나면 함수도 끝
            break
        end += 1
    print("─" * 78)
    print("{}  ::  {}()".format(path, name))
    print("─" * 78)
    print("\n".join(lines[start:end]).rstrip())


BUILDER_PY = find_file("../session2_index_builder.py", "session2_index_builder.py")
show_function(BUILDER_PY, "request_index")

# 이 컬렉션에 실제로 어떤 인덱스가 붙어 있는지 먼저 확인합니다.
# 인덱스가 없는 필드로 검색하면 kNN 완전탐색으로 처리되므로 아래 지연시간의 의미가 달라집니다.
indexes = list(service_client.list_indexes(parent=COLLECTION_NAME))
if indexes:
    for index in indexes:
        print("인덱스 {}  ← index_field={}  store_fields={}".format(
            index.name.split("/")[-1], index.index_field, list(index.store_fields)))
else:
    print("⚠️ 인덱스가 아직 없습니다. 아래 지연시간은 ANN이 아니라 kNN 완전탐색 수치입니다.")
print()

# 질의의 실제 지연시간을 측정합니다.
probe_vector = embed(text="wireless noise cancelling headphones")
latencies = []
for _ in range(5):
    started = perf_counter()
    vector_search(probe_vector, TEXT_FIELD, top_k=20)
    latencies.append((perf_counter() - started) * 1000)

print("\nsearch_ms 5회 :", ", ".join("{:.1f}".format(value) for value in latencies))
print("중앙값        : {:.1f} ms  (임베딩 생성 시간 제외, 순수 검색)".format(statistics.median(latencies)))

## 11. `app/embedding_vector.py` 소스 대조

노트북의 `hybrid_search()`와 앱의 `_hybrid_collection_search()`를 나란히 출력합니다. 절 순서·`weights`·결과 파싱까지 같은 요청이고 `top_k`(앱은 100)와 로깅만 다릅니다. 단, 이 앱 함수는 런타임에 호출되지 않습니다.

In [ ]:
import inspect

APP_EMBEDDING_PY = find_file(
    "app/embedding_vector.py",
    "../part2/app/embedding_vector.py",
)

show_function(APP_EMBEDDING_PY, "_hybrid_collection_search")
print()
print("─" * 78)
print("이 노트북의 hybrid_search()")
print("─" * 78)
print(inspect.getsource(hybrid_search).rstrip())

> [!IMPORTANT]
> ### 재현율 vs 지연 — 이 앱에서 실제로 내려진 결정
>
> `_hybrid_collection_search()`는 구현돼 있지만 실행되지 않습니다. `_collection_search()`가 텍스트 단독 경로로 단축되고 하이브리드 호출부는 주석 처리되어 있습니다 — 커밋 `4bc2657 "Set use only text for latency reduce"`의 의도적 변경입니다.
>
> - **잃은 것**: 크로스모달 재현율. `image_embedding` 쪽 후보를 통째로 보지 못합니다.
> - **얻은 것**: 서버측 벡터 검색 2회 → 1회. Live 음성 대화에서 수백 ms가 대화 흐름을 좌우하고, `find_items`는 쿼리를 여러 개 동시에 던집니다.
> - **무관**: 카메라 경로는 원래부터 `image_embedding` 단독입니다.
>
> 검색 설계는 기능을 전부 켜는 일이 아니라, 어느 재현율을 어느 지연에 팔지 고르는 일입니다.

In [ ]:
show_function(APP_EMBEDDING_PY, "_collection_search")

## 12. 에이전트 프롬프트와 `find_items` 툴 호출 흐름

```
경로 A — 카메라 프레임 (자동)
  JPEG → 유사상품 워커 스레드 → _image_similarity_search()
       → image_embedding 단독 검색 [6단계 ①] → 좌측 타일 실시간 갱신

경로 B — 음성 발화 → 툴 호출
  find_items(queries=[영어 쿼리 N개], ranking_query="영어 요약")
    → 쿼리별 스레드 병렬 _collection_search(text=q)   [5단계]
    → id 중복 제거 → _rank_results()                  [8단계]
    → 상위 64건 렌더링 + 음성 브리핑
```

프롬프트는 검색 쿼리를 **영어**로(상품 설명문이 영어), 음성 응답은 **한글**로 강제합니다.

In [ ]:
APP_PROMPT_PY = find_file(
    "app/prompt.py",
    "../part2/app/prompt.py",
)

prompt_source = APP_PROMPT_PY.read_text()
step1 = "## 1단계" + prompt_source.split("## 1단계", 1)[1].split("## 2단계", 1)[0]
print("─" * 78)
print("{}  ::  AGENT_PROMPT 발췌".format(APP_PROMPT_PY))
print("─" * 78)
print(step1.rstrip())

## 실습 2 완료 🎉

1. 질의 코드는 Part 1과 동일하고, 달라진 것은 컬렉션에 붙은 ScaNN 인덱스뿐이다.
2. 벡터 하나를 `text_embedding` / `image_embedding` 두 필드에 모두 던질 수 있다.
3. RRF `weights`가 Part 1의 `alpha`를 대체하고, 뒤집으면 순위가 바뀐다.
4. Ranking API가 재현율 위주 결과를 정밀도 위주로 다시 세운다.
5. 실제 서비스는 이 손잡이를 전부 켜지 않는다 — 검색 설계는 트레이드오프 선택이다.

`part2/README.md`로 돌아가 배포 상태를 확인하고 QR 코드로 에이전트를 사용해 봅니다.